In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/README.dataset.txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/README.roboflow.txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/README.md
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/valid/labels/DV (488).txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/valid/labels/DV (403).txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/valid/labels/BV (538).txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/valid/labels/BV (643).txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/valid/labels/BV (367).txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/valid/labels/DV (792).txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/valid/labels/DV (454).txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/valid/labels/DV (578).txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/valid/labels/BV (224).txt
/kaggle/input/datasets/truong9/bird-vs-drone/Dataset/valid/labels/DV (749).txt
/k

In [3]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 962.9 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 12.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 3.8 MB/s eta 0:00:00


In [10]:
import os
import yaml
from ultralytics import YOLO


dataset_base = '/kaggle/input/datasets/truong9/Bird vs Drone/Dataset'


if not os.path.exists(dataset_base):
    import glob
    matches = glob.glob('/kaggle/input/**/Dataset', recursive=True)
    if matches:
        dataset_base = matches[0]

print(f"Dataset base path located at: {dataset_base}")

yaml_path = '/kaggle/working/full_dataset.yaml'
yaml_content = {
    'path': dataset_base,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {
        0: 'bird',
        1: 'drone'
    }
}

with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f)

print(f"Successfully generated config file at: {yaml_path}")


model = YOLO("yolov8s.pt")


print("\nStarting 60-epoch full dataset training with anti-confusion parameters...")
results = model.train(
    data=yaml_path,
    epochs=60,          
    imgsz=640,
    batch=16,
    device=[0, 1],      
    cls=3.0,            
    mixup=0.1,         
    copy_paste=0.1,   
    cos_lr=True         
)

print("\n60-epoch training complete! Your model is fully trained on the entire dataset.")

Dataset base path located at: /kaggle/input/datasets/truong9/bird-vs-drone/Dataset
Successfully generated config file at: /kaggle/working/full_dataset.yaml

Starting 60-epoch full dataset training with anti-confusion parameters...
Ultralytics 8.4.163 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                        CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=3.0, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/60       2.3G      1.403      8.804      1.562         42        640: 100% ━━━━━━━━━━━━ 420/420 6.0it/s 1:100.2sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 55/55 7.4it/s 7.4s0.1s
                   all       1740       1782      0.346      0.336      0.363      0.127

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/60       2.3G      1.366       8.27      1.532         36        640: 100% ━━━━━━━━━━━━ 420/420 6.5it/s 1:040.1sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 55/55 7.7it/s 7.1s0.1s
                   all       1740       1782      0.338      0.313      0.331      0.137

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/60      2.31G      1.311      7.724      1.486         34        6

In [11]:
import os
from pathlib import Path
from ultralytics import YOLO


model_path = '/kaggle/working/runs/detect/train/weights/best.pt'
model = YOLO(model_path)


video_dir = Path('/kaggle/input/datasets/anuragpatel12/video-test')
output_dir = '/kaggle/working/video_predictions'
os.makedirs(output_dir, exist_ok=True)


video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.webm')


video_files = [f for f in video_dir.iterdir() if f.suffix.lower() in video_extensions]

if not video_files:
    print(f"No video files found in {video_dir}")
else:
    print(f"Found {len(video_files)} video(s) to process. Running inference...")
    
    for video_path in video_files:
        print(f"-> Processing: {video_path.name}")
        
       
        results = model.predict(
            source=str(video_path),
            conf=0.3,           
            save=True,          
            project=output_dir,
            name='results',    
            device=0,           
            stream=False       
        )

    print(f"\nAll videos processed successfully! Saved to: {output_dir}/results/")

Found 2 video(s) to process. Running inference...
-> Processing: Create_a_realistic_outdoor_surveillance_20260925225637.mp4

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/192) /kaggle/input/datasets/anuragpatel12/video-test/Create_a_realistic_outdoor_surveillance_20260925225637.mp4: 384x640 1 bird, 8.1ms
video 1/1 (frame 2/192) /kaggle/input/datasets/anuragpatel12/video-test/Create_a_realistic_outdoor_surveillance_20260925225637.mp4: 384x640 1 bird, 8.0ms
video 1/1 (frame 3/192) /ka

In [17]:
import os
import glob
import yaml
from ultralytics import YOLO


print("Searching for dataset path...")
search_results = glob.glob('/kaggle/input/**/test/images', recursive=True)

if not search_results:
  
    search_results = glob.glob('/kaggle/input/**/Dataset', recursive=True)

if not search_results:
    raise FileNotFoundError("Could not locate your dataset under /kaggle/input/. Make sure the dataset is added to your notebook!")


found_path = search_results[0]
if 'test/images' in found_path:
    dataset_base = found_path.split('test/images')[0].rstrip('/')
else:
    dataset_base = found_path

print(f"Successfully found dataset base path: {dataset_base}")


yaml_data = {
    'path': dataset_base,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {
        0: 'bird',
        1: 'drone'
    }
}

yaml_path = '/kaggle/working/temp_test.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_data, f)


model_weight_path = '/kaggle/working/runs/detect/train/weights/best.pt'
model = YOLO(model_weight_path)


print("\nRunning test evaluation and generating confusion matrix...")
metrics = model.val(
    data=yaml_path,
    split='test',  
    conf=0.001,
    iou=0.6,
    plots=True    
)

print("\n--- TEST RESULTS ---")
print(f"mAP50 (Overall): {metrics.box.map50:.4f}")
print(f"mAP50-95 (Overall): {metrics.box.map:.4f}")
print(f"Bird mAP50: {metrics.box.maps[0]:.4f}")
print(f"Drone mAP50: {metrics.box.maps[1]:.4f}")
print("\nCheck your latest /kaggle/working/runs/detect/val* folder for 'confusion_matrix.png'!")

Searching for dataset path...
Successfully found dataset base path: /kaggle/input/datasets/truong9/bird-vs-drone/Dataset

Running test evaluation and generating confusion matrix...
Ultralytics 8.4.163 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 11,126,358 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 50.0±15.0 MB/s, size: 29.9 KB)
val: Scanning /kaggle/input/datasets/truong9/bird-vs-drone/Dataset/test/labels... 889 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 889/889 789.9it/s 1.1s0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/truong9/bird-vs-drone/Dataset/test is not writable, cache not saved.
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 437, len(boxes) = 906. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
           